# Video Manipulation: Split Into N Equal Clips

Split `Artifacts/dashcam_main.mp4` into `N_CLIPS` equal-duration segments.

## Outputs

Clips are written to:

- `OUTPUT_DIR` (default: `Artifacts/video_clips`)

Naming format:

- `{OUTPUT_BASENAME}_clip_01_of_XX.mp4`

## Cutting mode

Two strategies are supported:

1) `STREAM_COPY = True` (fast)
- uses `ffmpeg -c copy`
- caveat: cut points may land on keyframes only, so boundaries can be approximate

2) `STREAM_COPY = False` (more predictable boundaries)
- re-encodes video (H.264/AAC)
- slower, but produces cut points closer to the requested timestamps

## Usage

1. Set `VIDEO_PATH` and `N_CLIPS`.
2. Set `STREAM_COPY` depending on your speed vs boundary accuracy needs.
3. Run the “Run split now” cell.

## Troubleshooting

- `ffmpeg` fails when `STREAM_COPY=True`: switch to `STREAM_COPY=False`.
- No output files created: verify `ffmpeg`/`ffprobe` are installed and `VIDEO_PATH` exists.


In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess
from typing import List



In [ ]:
def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "Code" else cwd

ROOT = _repo_root()

VIDEO_PATH = ROOT / "Artifacts" / "dashcam_main.mp4"

N_CLIPS = 5

OUTPUT_DIR = ROOT / "Artifacts" / "video_clips"
OUTPUT_BASENAME = "dashcam_main"

STREAM_COPY = True

OVERWRITE = True


In [ ]:
def _run(cmd: List[str]) -> subprocess.CompletedProcess[str]:
    """Run a subprocess command (raises on failure)."""
    return subprocess.run(
        cmd,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

def _require_ffmpeg_tools() -> None:
    """Ensure ffmpeg/ffprobe are available."""
    for tool in ("ffmpeg", "ffprobe"):
        try:
            _run([tool, "-version"])
        except Exception as e:
            raise RuntimeError(
                f"Required tool `{tool}` was not found or failed to run. "
                f"Install ffmpeg (includes ffprobe) and ensure it's on PATH. Details: {e}"
            )

def _probe_duration_seconds(video_path: Path) -> float:
    """Return total duration (seconds) using ffprobe."""
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration",
        "-of",
        "default=noprint_wrappers=1:nokey=1",
        str(video_path),
    ]
    proc = _run(cmd)
    raw = proc.stdout.strip()
    if not raw:
        raise RuntimeError(f"Could not read duration from ffprobe for: {video_path}")
    return float(raw)

def split_video_into_equal_clips(
    *,
    video_path: Path,
    n_clips: int,
    output_dir: Path,
    output_basename: str,
    stream_copy: bool = True,
    overwrite: bool = True,
) -> List[Path]:
    """Split `video_path` into `n_clips` equal-duration clips.

    Cut logic:
    - Uses ffprobe duration as the source of truth.
    - Clips are evenly spaced in time.
    - The last clip ends at the full probed duration.

    Notes on `stream_copy`:
    - When `stream_copy=True`, ffmpeg uses `-c copy`.
    - That is fast, but cut points may be constrained by keyframes, so segment boundaries
      can be approximate.
    - If stream-copy fails, this function retries by re-encoding.
    """
    if n_clips < 1:
        raise ValueError("N_CLIPS must be >= 1")

    _require_ffmpeg_tools()

    if not video_path.exists():
        raise FileNotFoundError(f"Input video not found: {video_path}")

    output_dir.mkdir(parents=True, exist_ok=True)

    duration = _probe_duration_seconds(video_path)
    if duration <= 0:
        raise RuntimeError(f"Invalid/probed duration for {video_path}: {duration}")

    clip_duration = duration / n_clips
    if clip_duration <= 0:
        raise RuntimeError("Computed clip duration is non-positive; check duration and N_CLIPS")

    print(f"Detected video duration: {duration:.3f}s")
    print(f"Splitting into {n_clips} clip(s), target clip length: {clip_duration:.3f}s")

    generated: List[Path] = []

    for i in range(n_clips):
        start = i * clip_duration
        if i == n_clips - 1:
            t = max(0.0, duration - start)
        else:
            t = clip_duration

        out_path = output_dir / (
            f"{output_basename}_clip_{i+1:02d}_of_{n_clips:02d}.mp4"
        )

        base_cmd: List[str] = [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            *( ["-y"] if overwrite else [] ),
            "-ss",
            str(start),
            "-i",
            str(video_path),
            "-t",
            str(t),
            "-map",
            "0",
        ]

        if stream_copy:
            encode_cmd = ["-c", "copy"]
        else:
            encode_cmd = [
                "-c:v",
                "libx264",
                "-preset",
                "medium",
                "-crf",
                "18",
                "-c:a",
                "aac",
                "-b:a",
                "192k",
                "-movflags",
                "+faststart",
            ]

        cmd = base_cmd + encode_cmd + [str(out_path)]

        print(f"\n[{i+1}/{n_clips}] Writing: {out_path.name}")
        print(f"  start={start:.3f}s  duration={t:.3f}s  stream_copy={stream_copy}")
        try:
            _run(cmd)
        except subprocess.CalledProcessError as e:
            if stream_copy:
                print(
                    f'ffmpeg stream-copy failed for {out_path.name}; '
                    f'retrying with re-encode. Error: {e}'
                )
                encode_cmd = [
                    '-c:v',
                    'libx264',
                    '-preset',
                    'medium',
                    '-crf',
                    '18',
                    '-c:a',
                    'aac',
                    '-b:a',
                    '192k',
                    '-movflags',
                    '+faststart',
                ]
                cmd_retry = base_cmd + encode_cmd + [str(out_path)]
                _run(cmd_retry)
            else:
                raise

        if out_path.exists() and out_path.stat().st_size > 0:
            generated.append(out_path)
        else:
            raise RuntimeError(
                f"ffmpeg reported success but output is missing/empty: {out_path}"
            )

    return generated


In [ ]:
generated_paths = split_video_into_equal_clips(
    video_path=VIDEO_PATH,
    n_clips=N_CLIPS,
    output_dir=OUTPUT_DIR,
    output_basename=OUTPUT_BASENAME,
    stream_copy=STREAM_COPY,
    overwrite=OVERWRITE,
)

print("\nDone.")
print(f"Generated {len(generated_paths)} file(s):")
for p in generated_paths:
    print(f"- {p}")
